In [3]:
### ARTS as the line-by-line code
import os
# set global variables BEFORE importing ARTS
os.environ['ARTS_DATA_PATH'] = '../arts-cat-data-2.6.14:../arts-xml-data-2.6.14'

import pyarts
from pyarts.arts import convert
# the fluxsim module
import FluxSimulator as fsm
import seaborn as sns


# helper packages
import numpy as np
import xarray as xr
from scipy.constants import speed_of_light as c
import matplotlib.pyplot as plt

import typhon
import typhon.constants as tpc
import pandas as pd

from scipy.optimize import minimize

from cycler import cycler

In [6]:
ddq_sw = xr.open_dataset('DDQ configurations/DDQ_SW.h5', engine = 'netcdf4')
ddq_sw


getfattr: /homedata/pczarnec/ddq-data-paper/DDQ: No such file or directory
getfattr: configurations/DDQ_SW.h5: No such file or directory


<xarray.Dataset> Size: 1kB
Dimensions:  (S: 64)
Coordinates:
  * S        (S) float64 512B 2.144e+03 2.411e+03 ... 4.426e+04 4.839e+04
Data variables:
    W        (S) float64 512B ...
Attributes:
    description:  64 optimized wavenumbers and weights, in the shortwave, opt...

In [ ]:
import pyarts
from scipy.interpolate import interp1d

# Your 64 frequencies (Hz)
f_grid = convert.kaycm2freq(ddq_sw.S.data)

solar = pyarts.xml.load("star/Sun/solar_spectrum_July_2008.xml")


# Frequencies in the XML (Hz)
f_sun = np.asarray(solar.grids[0])

# Solar spectrum values
I_sun = np.asarray(solar.data).squeeze()[:, 0]


interp = interp1d(
    f_sun,
    I_sun,
    kind="linear",
    bounds_error=False,
    fill_value="extrapolate",
)

I64 = interp(f_grid)

In [19]:
ddq_sw = ddq_sw.assign({"solar_source" : (("S"), I64)})
ddq_sw.to_netcdf("DDQ configurations/DDQ_SW_source.h5")